# Vietnamese GEC: Qwen3-4B QLoRA experiments

Clone a selected branch of **[rquq/GECC_VLSP_26](https://github.com/rquq/GECC_VLSP_26)** and execute from its repository root. Enable **Internet**, select a CUDA GPU (Kaggle **GPU T4 x2** is supported), and attach a **private** dataset containing `VGEC_balanced.json` built locally with `tools/build_dataset.py`. The selected branch must be pushed before running this notebook. Do not publish the dataset or credentials.

The selected JSON config controls model loading, adapters, training, prompt, generation, evaluation, seeds, and output. Seeds run sequentially on one GPU; no DDP. Validation comes from the original corpus; only training is audited and balanced. For a short runtime check, use `RUN_CONFIG_OVERRIDES` to limit rows and steps. Completed adapters and reports stay inside the cloned project under `runs/`, which Kaggle retains as notebook output. The public model needs no token; optional private/gated access uses Kaggle Secrets `HF_TOKEN`.



In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
import os
import subprocess
import sys

# The baseline uses PyTorch; avoid importing optional TensorFlow/Flax stacks.
os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("USE_FLAX", "0")

REPO_URL = "https://github.com/rquq/GECC_VLSP_26.git"
BRANCH = "rquq/test"  # Change to any published branch you want to train.
PROJECT_SUBDIR = "."  # The baseline lives at the root of this repository.
CONFIG_PATH = "configs/baseline.json"
RUN_CONFIG_OVERRIDES = {}  # Optional nested settings; leave empty to use JSON as-is.
USE_HF_SECRET = False
CLONE_DIR = Path("/kaggle/working/gecc-source")
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S-%f')
print("Branch:", BRANCH, "| config:", CONFIG_PATH)



In [ ]:
# This cell can be rerun after a kernel restart. Git runs as a subprocess.
# Reuse only a clean checkout of the same repository and branch.
from urllib.parse import urlsplit
if urlsplit(REPO_URL).username is not None:
    raise ValueError("Do not embed GitHub credentials in REPO_URL.")
subprocess.run(["git", "check-ref-format", "--branch", BRANCH], check=True, capture_output=True)
if CLONE_DIR.exists():
    if not (CLONE_DIR / ".git").is_dir():
        raise RuntimeError(f"{CLONE_DIR} exists but is not a Git clone; choose another CLONE_DIR.")
    existing_url = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=CLONE_DIR, text=True).strip()
    if existing_url != REPO_URL:
        raise RuntimeError("CLONE_DIR belongs to a different repository; choose another CLONE_DIR.")
    existing_branch = subprocess.check_output(["git", "branch", "--show-current"], cwd=CLONE_DIR, text=True).strip()
    if existing_branch != BRANCH:
        raise RuntimeError("CLONE_DIR uses a different branch; choose another CLONE_DIR.")
    changes = subprocess.check_output(["git", "status", "--porcelain"], cwd=CLONE_DIR, text=True)
    if changes.strip():
        raise RuntimeError("CLONE_DIR has local changes; preserve them and choose another CLONE_DIR.")

remote = subprocess.run(["git", "ls-remote", "--exit-code", "--heads", "--", REPO_URL,
                         f"refs/heads/{BRANCH}"], capture_output=True, text=True)
if remote.returncode == 2:
    raise RuntimeError(f"Branch {BRANCH!r} does not exist on GitHub. Push it or select a published BRANCH.")
if remote.returncode != 0:
    raise RuntimeError("Cannot read the Git repository. Enable Kaggle Internet and check repository access.")
if CLONE_DIR.exists():
    subprocess.run(["git", "fetch", "origin", f"refs/heads/{BRANCH}"], cwd=CLONE_DIR, check=True)
    subprocess.run(["git", "merge", "--ff-only", "FETCH_HEAD"], cwd=CLONE_DIR, check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--single-branch", "--branch", BRANCH,
                    "--", REPO_URL, str(CLONE_DIR)], check=True)
PROJECT_DIR = (CLONE_DIR / PROJECT_SUBDIR).resolve()
if not PROJECT_DIR.is_relative_to(CLONE_DIR.resolve()):
    raise ValueError("PROJECT_SUBDIR must stay inside the cloned repository.")
for filename in ["requirements.txt", "model/train.py", "model/config.py", "model/experiment.py", "tools/kaggle_run.py", CONFIG_PATH]:
    if not (PROJECT_DIR / filename).is_file():
        raise FileNotFoundError(f"Missing {filename} in {PROJECT_DIR}; check the selected branch and PROJECT_SUBDIR.")
SOURCE_INFO = {"repo_url": REPO_URL, "branch": BRANCH, "project_subdir": PROJECT_SUBDIR,
               "commit": subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=CLONE_DIR, text=True).strip()}
print(json.dumps(SOURCE_INFO, indent=2))



In [ ]:
# Keep the CUDA-enabled torch provided by Kaggle. A constraint prevents
# pip from replacing it while installing the pinned Hugging Face stack.
from importlib.metadata import version
from packaging.version import Version
import tempfile

installed_torch = version("torch")
if Version(installed_torch).base_version != "2.7.1":
    raise RuntimeError(f"requirements.txt targets the HiGANplus stack with CUDA PyTorch 2.7.1; found {installed_torch}. Use a matching image rather than replacing its CUDA build.")
packages = ["transformers", "peft", "accelerate", "bitsandbytes", "safetensors", "huggingface-hub", "tokenizers"]
imported_before = {name: version(name) for name in packages if name.replace("-", "_") in sys.modules}
with tempfile.TemporaryDirectory() as temporary:
    constraints = Path(temporary) / "constraints.txt"
    constraints.write_text(f"torch=={installed_torch}\n", encoding="utf-8")
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements.txt"),
                    "--constraint", str(constraints)], cwd=PROJECT_DIR, check=True)
changed_imports = [name for name, old in imported_before.items() if version(name) != old]
if changed_imports:
    raise RuntimeError(f"Restart the kernel to reload {changed_imports}, then rerun from the parameter cell; the clean clone is reused.")
# Check the training stack directly; unrelated packages in Kaggle's image
# can have pre-existing conflicts that do not affect this PyTorch baseline.
subprocess.run([sys.executable, "-c", "import torch, transformers, peft, accelerate, bitsandbytes; print('Training imports passed')"], check=True)
print("Pinned dependencies installed; retained torch", installed_torch)



In [ ]:
if USE_HF_SECRET:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

# The launcher validates data, dependencies, CUDA devices, and the Git commit
# before downloading weights. Each seed uses one GPU sequentially and independently.
config_file = (PROJECT_DIR / CONFIG_PATH).resolve()
run_config = json.loads(config_file.read_text(encoding="utf-8"))
def apply_overrides(target, overrides):
    for key, value in overrides.items():
        if isinstance(value, dict) and isinstance(target.get(key), dict):
            apply_overrides(target[key], value)
        else:
            target[key] = value
apply_overrides(run_config, RUN_CONFIG_OVERRIDES)
configured_root = Path(run_config["run"]["output_root"]).expanduser()
if not configured_root.is_absolute():
    configured_root = PROJECT_DIR / configured_root
OUTPUT_DIR = configured_root / f'{run_config["run"]["name"]}-{RUN_ID}'
if RUN_CONFIG_OVERRIDES:
    config_file = CLONE_DIR.parent / f"vgec-config-{RUN_ID}.json"
    config_file.write_text(json.dumps(run_config, ensure_ascii=False, indent=2), encoding="utf-8")
command = [sys.executable, str(PROJECT_DIR / "tools" / "kaggle_run.py"),
           "--config", str(config_file), "--output-dir", str(OUTPUT_DIR),
           "--expected-commit", SOURCE_INFO["commit"]]
subprocess.run(command + ["--preflight-only"], cwd=PROJECT_DIR, check=True)



In [ ]:
# Evaluate each fresh unadapted model, train response-only QLoRA, reload its best
# adapter, evaluate, record source provenance, and package all seed directories.
subprocess.run(command, cwd=PROJECT_DIR, check=True)



In [ ]:
from IPython.display import JSON, FileLink, display
for filename in ["comparison.json", "source_manifest.json"]:
    print(filename)
    display(JSON(json.loads((OUTPUT_DIR / filename).read_text(encoding="utf-8"))))
archive_path = Path(str(OUTPUT_DIR) + "-artifacts.zip")
if archive_path.is_file():
    display(FileLink(str(archive_path)))
for seed in run_config["run"]["seeds"]:
    print("Saved adapter:", OUTPUT_DIR / f"seed-{seed}" / "adapter")
print("Save a Kaggle notebook version to retain the ZIP and run directory under Output.")



## Saved output

Each run saves a resolved `config.json`, `comparison.json`, `source_manifest.json`, and a `seed-<seed>/` folder for each configured seed. Each seed stores its best adapter/tokenizer in `adapter/`, baseline and adapted predictions/metrics, identity metrics, epoch history, and run manifest. Adapter `inference.json` preserves the prompt and generation settings. `evaluation.save_epoch_adapters` controls additional epoch adapters; `run.archive` controls ZIP packaging. Outputs are private artifacts, not an official submission.

For another run, rerun all cells for a fresh output directory. The source cell fetches the same branch and refuses local code edits or a different repository/branch. For a branch switch or force-pushed branch, choose a new `CLONE_DIR`. Edit the JSON config or use `RUN_CONFIG_OVERRIDES` in the parameter cell. Predictions use a local proposed format; the competition's official submission format remains unconfirmed.
